# TP Intelligence Artificielle - Recherche Arborescente Informée

# **Partie 0 : Visualisation d'états, Classe Taquin et Classe Node**

Nous récupérons les classes du TP précédent ainsi que la fonction visualise_state.

In [1]:
from IPython.display import display, HTML

def visualize_state(state):
    """Visualizes the given state of the Taquin using HTML."""
    html = "<table>"
    for row in state:
        html += "<tr>"
        for tile in row:
            if tile == 0:
                html += "<td style='background-color: lightgray; width: 30px; height: 30px; text-align: center; font-size: 20px;'> </td>"  # Blank tile
            else:
                html += f"<td style='background-color: lightblue; width: 30px; height: 30px; text-align: center; font-size: 20px;'>{tile}</td>"
        html += "</tr>"
    html += "</table>"
    display(HTML(html))

class Taquin:
    """
    A class representing the Taquin problem.
    """

    def __init__(self, initial_state, goal_state, size):
        self.initial_state = initial_state
        self.goal_state = goal_state
        self.size = size

    def actions(self, state):
        """Returns the possible actions (moves) from the given state."""
        # Find the position of the blank tile (0)
        row, col = next(
            (r, c)
            for r, row in enumerate(state)
            for c, val in enumerate(row)
            if val == 0
        )

        # Define possible moves (up, down, left, right)
        possible_actions = []
        if row > 0:
            possible_actions.append("up")
        if row < self.size-1:
            possible_actions.append("down")
        if col > 0:
            possible_actions.append("left")
        if col < self.size-1:
            possible_actions.append("right")

        return possible_actions

    def result(self, state, action):
        """Returns the state that results from applying the given action."""
        # Create a copy of the state to avoid modifying the original
        new_state = [list(row) for row in state]

        # Find the position of the blank tile (0)
        row, col = next(
            (r, c)
            for r, row in enumerate(state)
            for c, val in enumerate(row)
            if val == 0
        )

        # Apply the action to move the blank tile
        if action == "up":
            new_state[row][col], new_state[row - 1][col] = (
                new_state[row - 1][col],
                new_state[row][col],
            )
        elif action == "down":
            new_state[row][col], new_state[row + 1][col] = (
                new_state[row + 1][col],
                new_state[row][col],
            )
        elif action == "left":
            new_state[row][col], new_state[row][col - 1] = (
                new_state[row][col - 1],
                new_state[row][col],
            )
        elif action == "right":
            new_state[row][col], new_state[row][col + 1] = (
                new_state[row][col + 1],
                new_state[row][col],
            )

        return new_state

    def is_goal(self, state):
        return state == self.goal_state  # Directly compare with goal_state

    def cost(self, state, action):
        return 1  # Default cost is 1

class Node:
    """
    A node in a search tree.
    __init__: Initializes a node with its state, parent, action, path cost, and depth.
    __repr__: Provides a string representation of the node.
    __lt__: Defines a comparison operator for nodes based on their states.
    expand: Generates child nodes by applying all possible actions.
    child_node: Creates a single child node for a given action.
    solution: Returns the sequence of actions that led to this node.
    path: Returns the path from the root to this node as a list of nodes.
    """

    def __init__(self, state, parent=None, action=None, path_cost = 0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost
        self.depth = 0 if parent is None else parent.depth + 1

    # def __repr__(self):
    #     return "<Node {}>".format(self.state)

    # def __lt__(self, other):
    #     return self.state < other.state

    def expand(self, problem):
        """List the nodes reachable in one step from this node."""
        return [
            self.child_node(problem, action)
            for action in problem.actions(self.state)
        ]

    def child_node(self, problem, action):
        """Create a child node by applying the given action."""
        next_state = problem.result(self.state, action)
        next_node = Node(
            next_state,
            parent=self,
            action=action,
            path_cost = self.path_cost + problem.cost(self.state, action),
        )
        return next_node

    def solution(self):
        """Return the sequence of actions to go from the root to this node."""
        return [node.action for node in self.path()[1:]]

    def path(self):
        """Return a list of nodes forming the path from the root to this node."""
        node, path_back = self, []
        while node:
            path_back.append(node)
            node = node.parent
        return list(reversed(path_back))

# **Partie 1 : Heuristiques pour le jeu de Taquin**

## **1.1 H$_1$ : Nombre de tuiles mal placées**

### **Exercice 1**

Écrivez une fonction qui calcule le nombre de tuiles mal placées dans un état du jeu de Taquin

In [2]:

def misplaced_tiles(state, goal_state):
    """H1: nombre de tuiles mal placees (sans compter la case vide 0)."""
    misplaced = 0
    for i in range(len(state)):
        for j in range(len(state[0])):
            if state[i][j] != 0 and state[i][j] != goal_state[i][j]:
                misplaced += 1
    return misplaced


### **Exercice 2**

Calculez le nombre de tuiles mal placées pour les états suivants :
1. [[1, 2, 3], [4, 5, 0], [6, 7, 8]]
2. [[1, 2, 3], [0, 5, 6], [4, 7, 8]]    
3. [[1, 0, 3], [4, 2, 5], [7, 8, 6]]
4. [[1, 0, 3], [4, 5, 2], [7, 6, 8]]
5. [[1, 0, 3], [4, 2, 5], [6, 7, 8]]
6. [[1, 2, 3], [4, 0, 6], [7, 5, 8]]
7. [[1, 2, 3], [0, 4, 6], [7, 5, 8]]
8. [[1, 2, 3], [4, 6, 0], [7, 5, 8]]
9. [[1, 3, 6], [4, 2, 5], [7, 0, 8]]  
10. [[1, 3, 6], [4, 2, 0], [7, 5, 8]]  
11. [[1, 3, 6], [4, 0, 2], [7, 5, 8]]  
12. [[3, 1, 2], [4, 6, 5], [7, 0, 8]]  
13. [[8, 1, 2], [0, 4, 3], [7, 6, 5]]
14. [[1, 4, 2], [7, 0, 6], [5, 3, 8]]
15. [[2, 8, 3], [1, 6, 4], [7, 0, 5]]

In [3]:

goal_state = [[1, 2, 3], [4, 5, 6], [7, 8, 0]]
states_ex2 = [
    [[1, 2, 3], [4, 5, 0], [6, 7, 8]],
    [[1, 2, 3], [0, 5, 6], [4, 7, 8]],
    [[1, 0, 3], [4, 2, 5], [7, 8, 6]],
    [[1, 0, 3], [4, 5, 2], [7, 6, 8]],
]

print("Valeurs H1 (tuiles mal placees):")
for i, state in enumerate(states_ex2, start=1):
    print(f"Etat {i}: H1 = {misplaced_tiles(state, goal_state)}")


Valeurs H1 (tuiles mal placees):
Etat 1: H1 = 3
Etat 2: H1 = 3
Etat 3: H1 = 3
Etat 4: H1 = 3


## **1.2 H$_2$ : Distance de Manhattan**

### **Exercice 3**
Écrivez une fonction qui calcule la distance de Manhattan entre un état et l'état but. La fonction **next()** pourrait être utile pour trouver la position d'une tuile donnée dans l'état but.

In [4]:

def manhattan_distance(state, goal_state):
    """H2: somme des distances de Manhattan de chaque tuile vers sa position but."""
    goal_positions = {
        goal_state[i][j]: (i, j)
        for i in range(len(goal_state))
        for j in range(len(goal_state[0]))
    }

    total = 0
    for i in range(len(state)):
        for j in range(len(state[0])):
            tile = state[i][j]
            if tile != 0:
                gi, gj = goal_positions[tile]
                total += abs(i - gi) + abs(j - gj)
    return total


### **Exercice 4**
1. Calculez la distance de Manhattan des états de l'exercice 2.
2. Que pouvez-vous observer par rapport aux deux heuristiques ?

In [5]:
print("Comparaison H1 vs H2 sur les etats de l'exercice 2:")
for i, state in enumerate(states_ex2, start=1):
    h1 = misplaced_tiles(state, goal_state)
    h2 = manhattan_distance(state, goal_state)
    print(f"Etat {i}: H1 = {h1}, H2 = {h2}")

print()
print("Observation:")
print("H2 (Manhattan) est en general plus informative que H1, car elle mesure combien de cases chaque tuile doit encore parcourir.")


Comparaison H1 vs H2 sur les etats de l'exercice 2:
Etat 1: H1 = 3, H2 = 5
Etat 2: H1 = 3, H2 = 3
Etat 3: H1 = 3, H2 = 3
Etat 4: H1 = 3, H2 = 5

Observation:
H2 (Manhattan) est en general plus informative que H1, car elle mesure combien de cases chaque tuile doit encore parcourir.


# **Partie 2 : Algorithme A étoile**

### **Exercice 5**

Créer une fonction A_etoile qui prend un objet problem en entrée ainsi qu'une heuristique et exécute l'algorithme A.*

Votre fonction doit :

1. Retourner le nœud objectif trouvé ainsi que le nombre de nœuds explorés.
2. Retourner None si l'algorithme explore tout l'arbre sans trouver le nœud objectif.
3. Prendre en compte un budget d'exploration pour arrêter l'algorithme si aucune solution n'est trouvée après plusieurs itérations (utile pour les grands problèmes). Le budget sera également un paramètre d'entrée de la fonction, avec float('inf') comme valeur par défaut.

Pour la **frontière** et l'ensemble des **nœuds déjà explorés**, vous pouvez utiliser des listes. Cependant, la frontière doit contenir les nœuds à explorer ainsi que leurs coûts (coût réel + heuristique). Pour trouver l'élément (nœud, coût) de coût minimal dans la frontière, effectuez une boucle **for** pour trouver l'indice, puis utilisez pop.

Pour ajouter un élément à une liste, vous pouvez utiliser **liste.append(element)**.

Rappelez-vous que la vérification **is_goal** doit être effectuée avant d'explorer les enfants.

In [6]:

import heapq
import itertools
import time


def state_to_tuple(state):
    return tuple(tuple(row) for row in state)


def A_etoile(problem, heuristic, max_explorations=20000):
    """A* avec file de priorite et dictionnaire des meilleurs couts g."""
    root = Node(problem.initial_state)
    root_key = state_to_tuple(root.state)

    frontier = []
    counter = itertools.count()

    root_h = heuristic(root.state, problem.goal_state)
    heapq.heappush(frontier, (root.path_cost + root_h, root_h, next(counter), root))

    best_g = {root_key: 0}
    num_explorations = 0

    while frontier and num_explorations < max_explorations:
        _, _, _, node = heapq.heappop(frontier)
        node_key = state_to_tuple(node.state)

        # Ignore une entree obsolete de la file de priorite
        if node.path_cost > best_g.get(node_key, float('inf')):
            continue

        num_explorations += 1

        if problem.is_goal(node.state):
            return node, num_explorations

        for child in node.expand(problem):
            child_key = state_to_tuple(child.state)
            g = child.path_cost

            if g < best_g.get(child_key, float('inf')):
                best_g[child_key] = g
                h = heuristic(child.state, problem.goal_state)
                f = g + h
                heapq.heappush(frontier, (f, h, next(counter), child))

    return None, num_explorations


### **Exercice 6**

Appliquez votre fonction A_etoile pour résoudre un Taquin 3x3 avec l'état initial [[1, 2, 3], [4, 5, 6], [0, 7, 8]] et les deux heuristiques.

Imprimez :
1. Le nombre de nœuds explorés par l'algorithme.
2. Le chemin permettant de passer de l'état initial à l'état objectif (la liste des actions effectuées).
3. Que pouvez-vous observer entre les deux heuristiques ?

In [7]:
initial_state = [[1, 2, 3], [4, 5, 6], [0, 7, 8]]
goal_state = [[1, 2, 3], [4, 5, 6], [7, 8, 0]]
problem = Taquin(initial_state, goal_state, size=3)

for name, heuristic in [("A*H1", misplaced_tiles), ("A*H2", manhattan_distance)]:
    start = time.time()
    goal_node, num_explorations = A_etoile(problem, heuristic, max_explorations=20000)
    elapsed = time.time() - start

    print()
    print(name)
    print(f"Nœuds explorés: {num_explorations}")
    print(f"Temps: {elapsed:.4f} s")

    if goal_node is None:
        print("Aucune solution trouvée.")
    else:
        print(f"Longueur du chemin: {len(goal_node.solution())}")
        print(f"Actions: {goal_node.solution()}")



A*H1
Nœuds explorés: 3
Temps: 0.0001 s
Longueur du chemin: 2
Actions: ['right', 'right']

A*H2
Nœuds explorés: 3
Temps: 0.0001 s
Longueur du chemin: 2
Actions: ['right', 'right']


# **Partie 3 : Comparaison empirique de DFS, BFS, A etoile avec H1 et A etoile avec H2**

Nous allons comparer les quatre algorithmes sur les 15 instances du jeu de Taquin de l'exercice 2. Pour cela, voici les résultats de BFS et DFS sur ces 15 instances. Attention au format des tableaux. Les noms des colonnes peuvent différer de ceux que vous avez utilisés. De plus, un path_length égal à -1 signifie que l'algorithme n'a pas trouvé la solution avant d'atteindre le budget d'exploration.

In [8]:
results_BFS = [
{'initial_state': [[1, 2, 3], [4, 5, 0], [6, 7, 8]],'algorithm': 'breadth_first_search', 'num_explorations': 1846, 'path_length': 13, 'execution_time': 0.2773609161376953},
{'initial_state': [[1, 2, 3], [0, 5, 6], [4, 7, 8]], 'algorithm': 'breadth_first_search', 'num_explorations': 6, 'path_length': 3, 'execution_time': 0.0},
{'initial_state': [[1, 0, 3], [4, 2, 5], [7, 8, 6]], 'algorithm': 'breadth_first_search', 'num_explorations': 7, 'path_length': 3, 'execution_time': 0.0},
{'initial_state': [[1, 0, 3], [4, 5, 2], [7, 6, 8]], 'algorithm': 'breadth_first_search', 'num_explorations': 10911, 'path_length': 17, 'execution_time': 8.975327491760254},
{'initial_state': [[1, 0, 3], [4, 2, 5], [6, 7, 8]], 'algorithm': 'breadth_first_search', 'num_explorations': 4960, 'path_length': 15, 'execution_time': 1.831636905670166},
{'initial_state': [[1, 2, 3], [4, 0, 6], [7, 5, 8]], 'algorithm': 'breadth_first_search', 'num_explorations': 3, 'path_length': 2, 'execution_time': 0.0},
{'initial_state': [[1, 2, 3], [0, 4, 6], [7, 5, 8]], 'algorithm': 'breadth_first_search', 'num_explorations': 8, 'path_length': 3, 'execution_time': 0.0},
{'initial_state': [[1, 2, 3], [4, 6, 0], [7, 5, 8]], 'algorithm': 'breadth_first_search', 'num_explorations': 8, 'path_length': 3, 'execution_time': 0.0},
{'initial_state': [[1, 3, 6], [4, 2, 5], [7, 0, 8]], 'algorithm': 'breadth_first_search', 'num_explorations': 20000, 'path_length': -1, 'execution_time': 36.05873966217041},
{'initial_state': [[1, 3, 6], [4, 2, 0], [7, 5, 8]], 'algorithm': 'breadth_first_search', 'num_explorations': 20, 'path_length': 5, 'execution_time': 0.0},
{'initial_state': [[1, 3, 6], [4, 0, 2], [7, 5, 8]], 'algorithm': 'breadth_first_search', 'num_explorations': 62, 'path_length': 6, 'execution_time': 0.0019948482513427734},
{'initial_state': [[3, 1, 2], [4, 6, 5], [7, 0, 8]], 'algorithm': 'breadth_first_search', 'num_explorations': 20000, 'path_length': -1, 'execution_time': 40.92391228675842},
{'initial_state': [[8, 1, 2], [0, 4, 3], [7, 6, 5]], 'algorithm': 'breadth_first_search', 'num_explorations': 20000, 'path_length': -1, 'execution_time': 40.67150044441223},
{'initial_state': [[1, 4, 2], [7, 0, 6], [5, 3, 8]], 'algorithm': 'breadth_first_search', 'num_explorations': 3321, 'path_length': 14, 'execution_time': 0.8201050758361816},
{'initial_state': [[2, 8, 3], [1, 6, 4], [7, 0, 5]], 'algorithm': 'breadth_first_search', 'num_explorations': 20000, 'path_length': -1, 'execution_time': 42.47766828536987}
]
results_DFS = [
{'initial_state': [[1, 2, 3], [4, 5, 0], [6, 7, 8]], 'algorithm': 'depth_first_search', 'num_explorations': 20000, 'path_length': -1, 'execution_time': 69.51113247871399},
{'initial_state': [[1, 2, 3], [0, 5, 6], [4, 7, 8]], 'algorithm': 'depth_first_search', 'num_explorations': 27, 'path_length': 27, 'execution_time': 0.0},
{'initial_state': [[1, 0, 3], [4, 2, 5], [7, 8, 6]], 'algorithm': 'depth_first_search', 'num_explorations': 20000, 'path_length': -1, 'execution_time': 60.40420460700989},
{'initial_state': [[1, 0, 3], [4, 5, 2], [7, 6, 8]], 'algorithm': 'depth_first_search', 'num_explorations': 20000, 'path_length': -1, 'execution_time': 62.47419261932373},
{'initial_state': [[1, 0, 3], [4, 2, 5], [6, 7, 8]], 'algorithm': 'depth_first_search', 'num_explorations': 20000, 'path_length': -1, 'execution_time': 67.59605073928833},
{'initial_state': [[1, 2, 3], [4, 0, 6], [7, 5, 8]], 'algorithm': 'depth_first_search', 'num_explorations': 412, 'path_length': 406, 'execution_time': 0.02211737632751465},
{'initial_state': [[1, 2, 3], [0, 4, 6], [7, 5, 8]], 'algorithm': 'depth_first_search', 'num_explorations': 20000, 'path_length': -1, 'execution_time': 65.1717848777771},
{'initial_state': [[1, 2, 3], [4, 6, 0], [7, 5, 8]], 'algorithm': 'depth_first_search', 'num_explorations': 886, 'path_length': 871, 'execution_time': 0.08270859718322754},
{'initial_state': [[1, 3, 6], [4, 2, 5], [7, 0, 8]], 'algorithm': 'depth_first_search', 'num_explorations': 20000, 'path_length': -1, 'execution_time': 66.5749568939209},
{'initial_state': [[1, 3, 6], [4, 2, 0], [7, 5, 8]], 'algorithm': 'depth_first_search', 'num_explorations': 20000, 'path_length': -1, 'execution_time': 65.61762142181396},
{'initial_state': [[1, 3, 6], [4, 0, 2], [7, 5, 8]], 'algorithm': 'depth_first_search', 'num_explorations': 20000, 'path_length': -1, 'execution_time': 66.86844420433044},
{'initial_state': [[3, 1, 2], [4, 6, 5], [7, 0, 8]], 'algorithm': 'depth_first_search', 'num_explorations': 20000, 'path_length': -1, 'execution_time': 80.54781603813171},
{'initial_state': [[8, 1, 2], [0, 4, 3], [7, 6, 5]], 'algorithm': 'depth_first_search', 'num_explorations': 20000, 'path_length': -1, 'execution_time': 75.41818499565125},
{'initial_state': [[1, 4, 2], [7, 0, 6], [5, 3, 8]], 'algorithm': 'depth_first_search', 'num_explorations': 20000, 'path_length': -1, 'execution_time': 69.51109766960144},
{'initial_state': [[2, 8, 3], [1, 6, 4], [7, 0, 5]], 'algorithm': 'depth_first_search', 'num_explorations': 20000, 'path_length': -1, 'execution_time': 83.47795271873474}
]

### **Exercice 7**
Utilisez (et adaptez si nécessaire) votre fonction compare_algorithms du premier TP pour résoudre tous les jeux avec les deux versions de l'algorithme A etoile.

In [9]:

def compare_algorithms(initial_states, goal_state, size=3, max_explorations=20000):
    """Compare A* avec H1 et H2 sur une liste d'etats initiaux."""
    results = []
    variants = [
        ('A*H1', misplaced_tiles),
        ('A*H2', manhattan_distance),
    ]

    for init_state in initial_states:
        for algo_name, heuristic in variants:
            problem = Taquin(init_state, goal_state, size=size)
            start = time.time()
            goal_node, num_explorations = A_etoile(problem, heuristic, max_explorations=max_explorations)
            elapsed = time.time() - start

            results.append({
                'initial_state': init_state,
                'algorithm': algo_name,
                'num_explorations': num_explorations,
                'path_length': len(goal_node.solution()) if goal_node else -1,
                'execution_time': elapsed,
            })

    return results


# On reutilise les 15 etats fournis dans les resultats BFS
initial_states = [entry['initial_state'] for entry in results_BFS]
goal_state = [[1, 2, 3], [4, 5, 6], [7, 8, 0]]

results_a_star = compare_algorithms(initial_states, goal_state, size=3, max_explorations=20000)
results_misplace = [r for r in results_a_star if r['algorithm'] == 'A*H1']
results_manhattan = [r for r in results_a_star if r['algorithm'] == 'A*H2']

print("Comparaison A*H1 vs A*H2 terminée.")


Comparaison A*H1 vs A*H2 terminée.


#### **Utilisez le code ci-dessous pour visualiser vos résultats et commentez**

In [10]:
import pandas as pd
import numpy as np
from tabulate import tabulate ## La librairie Tabulate doit être installée. Si vous ne l'avez pas, vous pouvez simplement utiliser "print(df)" pour visualiser le tableau.

Data = []
for i in range(len(initial_states)):
    Data.append([int(i+1),
                 results_BFS[i]['num_explorations'], results_DFS[i]['num_explorations'], results_misplace[i]['num_explorations'], results_manhattan[i]['num_explorations'],
                 results_BFS[i]['path_length'], results_DFS[i]['path_length'], results_misplace[i]['path_length'], results_manhattan[i]['path_length'],
                 round(results_BFS[i]['execution_time'],2), round(results_DFS[i]['execution_time'],2), round(results_misplace[i]['execution_time'],2), round(results_manhattan[i]['execution_time'],2)])

df = pd.DataFrame(Data, columns=["State", "Nodes explored BFS", "Nodes explored DFS", "Nodes explored A*H1", "Nodes explored A*H2",
                                 "Path length BFS", "Path length DFS", "Path length A*H1", "Path length A*H2",
                                 "Execution Time BFS", "Execution Time DFS","Execution Time A*H1", "Execution Time A*H2"])
print(tabulate(df, headers='keys', tablefmt='pretty'))

+----+-------+--------------------+--------------------+---------------------+---------------------+-----------------+-----------------+------------------+------------------+--------------------+--------------------+---------------------+---------------------+
|    | State | Nodes explored BFS | Nodes explored DFS | Nodes explored A*H1 | Nodes explored A*H2 | Path length BFS | Path length DFS | Path length A*H1 | Path length A*H2 | Execution Time BFS | Execution Time DFS | Execution Time A*H1 | Execution Time A*H2 |
+----+-------+--------------------+--------------------+---------------------+---------------------+-----------------+-----------------+------------------+------------------+--------------------+--------------------+---------------------+---------------------+
| 0  |  1.0  |       1846.0       |      20000.0       |        144.0        |        57.0         |      13.0       |      -1.0       |       13.0       |       13.0       |        0.28        |       69.51        | 


## Corrige detaille - Heuristiques et A*

### Pourquoi H2 (Manhattan) est souvent meilleure que H1
- H1 dit seulement si une tuile est bien placee ou non.
- H2 mesure la distance restante de chaque tuile vers sa position finale.
- Donc H2 discrimine mieux les etats et guide mieux la recherche.

### Lien entre Dijkstra et A*
- Dijkstra = A* avec h(n)=0 pour tous les noeuds.
- A* devient plus rapide si l'heuristique est admissible et informative.

### Conditions importantes
- Heuristique admissible: n surestime jamais le cout reel restant.
- Heuristique consistante: h(n) <= cout(n,n') + h(n').



## FAQ rapide (TP2)

**Q1. Pourquoi A* garde `best_g` ?**
Pour ignorer les chemins plus chers vers un etat deja atteint.

**Q2. A* garantit-il l'optimalite ?**
Oui, si l'heuristique est admissible (et idealement consistante).

**Q3. Quand utiliser Dijkstra ?**
Quand on n'a pas d'heuristique fiable ou sur des graphes generaux avec couts positifs.
